# FinBERT vs DistilBERT on Financial PhraseBank
Same data, same settings, 3 seeds each. The test set is used once, only for the final scores (no tuning on it).
Run on a GPU: Runtime > Change runtime type > T4 GPU.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn matplotlib

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments, set_seed)

raw = load_dataset("FinanceMTEB/financial_phrasebank")
print(raw)
labels_train = np.array(raw["train"]["label"])
print("train label counts:", np.bincount(labels_train), "| test label counts:", np.bincount(raw["test"]["label"]))
print(dict(zip(raw["train"]["label"][:50], raw["train"]["label_text"][:50])))

Check the printed mapping above. The code below assumes 0 = negative, 1 = neutral, 2 = positive. If your mapping differs, change `NAMES`.

In [ ]:
NAMES = ["negative", "neutral", "positive"]
class_weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=labels_train)
weights = torch.tensor(class_weights, dtype=torch.float)
print("class weights:", class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def run(model_name, seed, epochs=4):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = lambda b: tok(b["text"], truncation=True, padding="max_length", max_length=128)
    cols = ["input_ids", "attention_mask", "labels"]
    tr = raw["train"].map(enc, batched=True).rename_column("label", "labels")
    te = raw["test"].map(enc, batched=True).rename_column("label", "labels")
    tr.set_format("torch", columns=cols)
    te.set_format("torch", columns=cols)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3,
        id2label=dict(enumerate(NAMES)), label2id={n: i for i, n in enumerate(NAMES)})
    args = TrainingArguments(
        output_dir=f"out_{seed}", num_train_epochs=epochs, learning_rate=2e-5,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        weight_decay=0.01, seed=seed, save_strategy="no", report_to="none",
        fp16=torch.cuda.is_available())
    trainer = WeightedTrainer(model=model, args=args, train_dataset=tr)
    trainer.train()
    preds = trainer.predict(te).predictions.argmax(-1)
    y = np.array(raw["test"]["label"])
    return {"acc": accuracy_score(y, preds), "f1": f1_score(y, preds, average="macro")}, y, preds

In [ ]:
MODELS = {"FinBERT": "ProsusAI/finbert", "DistilBERT": "distilbert-base-uncased"}
SEEDS = [42, 43, 44]
results, last_preds = {}, {}
for label, name in MODELS.items():
    runs = []
    for s in SEEDS:
        r, y, p = run(name, s)
        runs.append(r)
        print(label, "seed", s, r)
    results[label] = runs
    last_preds[label] = p

In [ ]:
summary = {}
print("| Model | Accuracy (mean +/- std) | Macro F1 (mean +/- std) |")
print("|---|---|---|")
for label, runs in results.items():
    a = np.array([r["acc"] for r in runs]); f = np.array([r["f1"] for r in runs])
    summary[label] = {"acc_mean": a.mean(), "acc_std": a.std(), "f1_mean": f.mean(), "f1_std": f.std(),
                      "runs": runs}
    print(f"| {label} | {a.mean():.3f} +/- {a.std():.3f} | {f.mean():.3f} +/- {f.std():.3f} |")
json.dump(summary, open("results.json", "w"), indent=2)

y = np.array(raw["test"]["label"])
for label in MODELS:
    print()
    print(label, "(last seed)")
    print(classification_report(y, last_preds[label], target_names=NAMES, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, label in zip(axes, MODELS):
    cm = confusion_matrix(y, last_preds[label])
    ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{label} (last seed)")
    ax.set_xticks(range(3)); ax.set_xticklabels(NAMES, rotation=30)
    ax.set_yticks(range(3)); ax.set_yticklabels(NAMES)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
# Download the outputs
from google.colab import files
files.download("results.json")
files.download("confusion_matrices.png")